# IHMM vs I-DT: correctness test

In [1]:
import numpy as np

from pymovements.events.detection.ihmm import ihmm
from pymovements.events.detection.idt import idt
from pymovements.synthetic import step_function
from pymovements.gaze.transforms_numpy import pos2vel

sampling_rate = 1000.0


## Test harness

Small set of helpers used by every test case below: extracting a plain table from whatever `ihmm`/`idt` return, counting events, pulling out `(onset, offset)` spans, computing overlap between spans, and a tiny pass/fail recorder so we get a summary table at the end of the notebook.

In [2]:
test_results = []


def record(name, passed, detail=""):
    test_results.append({"test": name, "passed": bool(passed), "detail": detail})
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}" + (f" — {detail}" if detail else ""))


def to_frame(events):
    """Return the underlying table (Polars/Pandas-like) for an events result."""
    return events.frame if hasattr(events, "frame") else events


def _column(frame, col):
    series = frame[col]
    return series.to_list() if hasattr(series, "to_list") else list(series)


def n_events(events):
    return len(to_frame(events))


def spans(events):
    """List of (onset, offset) tuples for every detected event."""
    frame = to_frame(events)
    if len(frame) == 0:
        return []
    onsets = _column(frame, "onset")
    offsets = _column(frame, "offset")
    return list(zip(onsets, offsets))


def show(label, events):
    print(f"--- {label} ({n_events(events)} event(s)) ---")
    display(to_frame(events))


def overlap_ratio(span_a, span_b):
    """Intersection-over-union of two (onset, offset) intervals."""
    a0, a1 = span_a
    b0, b1 = span_b
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    return inter / union if union > 0 else 0.0


def best_overlaps(spans_a, spans_b):
    """For each span in a, the best IoU achievable against any span in b."""
    if not spans_a:
        return []
    return [max((overlap_ratio(sa, sb) for sb in spans_b), default=0.0) for sa in spans_a]


def assert_event_count(name, events, expected):
    actual = n_events(events)
    passed = actual == expected
    record(f"{name}: event count == {expected}", passed, f"got {actual}")
    assert passed, f"{name}: expected {expected} event(s), got {actual}"


def assert_algorithms_agree(name, ihmm_events, idt_events, min_overlap=0.5):
    """Same number of fixations, and each ihmm fixation overlaps some idt fixation well."""
    ihmm_spans, idt_spans = spans(ihmm_events), spans(idt_events)
    count_match = len(ihmm_spans) == len(idt_spans)
    record(
        f"{name}: fixation counts match",
        count_match,
        f"ihmm={len(ihmm_spans)}, idt={len(idt_spans)}",
    )
    overlaps = best_overlaps(ihmm_spans, idt_spans)
    worst = min(overlaps) if overlaps else 1.0
    ok = worst >= min_overlap
    record(f"{name}: min pairwise overlap >= {min_overlap}", ok, f"worst overlap={worst:.2f}")
    return count_match and ok


## Case 1 — No fixations

Every single sample jumps to a wildly different position (`length=6`, one step per sample). Since consecutive samples never stay close together for more than one sample, there is no window over which either a velocity or a dispersion threshold can be satisfied — **neither algorithm should report any fixation.**

In [3]:
positions_no_fix = step_function(
    length=6,
    steps=[1, 2, 3, 4, 5],
    values=[
        (100., 100.),
        (-100., -100.),
        (100., -100.),
        (-100., 100.),
        (200., 0.),
    ],
    start_value=(0., 0.),
)

velocities_no_fix = pos2vel(positions_no_fix, sampling_rate=sampling_rate)

ihmmEvents = ihmm(velocities_no_fix, reestimation=True, name="ihmm_fix")
idtEvents = idt(positions_no_fix, name="idt_fix")

show("IHMM — no fixations", ihmmEvents)
show("IDT — no fixations", idtEvents)


--- IHMM — no fixations (0 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64


--- IDT — no fixations (0 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64


In [4]:
assert_event_count("No fixations / ihmm", ihmmEvents, 0)
assert_event_count("No fixations / idt", idtEvents, 0)


[PASS] No fixations / ihmm: event count == 0 — got 0
[PASS] No fixations / idt: event count == 0 — got 0


## Case 2 — One fixation

The trace starts at `(0, 0)` for a single sample, then jumps once to `(50, 50)` and stays there for the rest of the recording. A one-sample segment cannot itself be identified as a fixation by any dispersion/velocity based method (there's no consecutive-sample window to measure), so **there should be exactly one fixation**, spanning almost the entire recording after the single opening sample.

In [5]:
length_one_fix = 200
positions_one_fix = step_function(
    length=length_one_fix,
    steps=[1],
    values=[(50., 50.)],
    start_value=(0., 0.),
)

velocities_one_fix = pos2vel(positions_one_fix, sampling_rate=sampling_rate)

ihmmEvents = ihmm(velocities_one_fix, reestimation=True, name="ihmm_fix")
idtEvents = idt(positions_one_fix, name="idt_fix")

show("IHMM — one fixation", ihmmEvents)
show("IDT — one fixation", idtEvents)


--- IHMM — one fixation (1 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""ihmm_fix""",3,199,196


--- IDT — one fixation (1 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""idt_fix""",1,199,198


In [6]:
assert_event_count("One fixation / ihmm", ihmmEvents, 1)
assert_event_count("One fixation / idt", idtEvents, 1)
assert_algorithms_agree("One fixation", ihmmEvents, idtEvents, min_overlap=0.9)


[PASS] One fixation / ihmm: event count == 1 — got 1
[PASS] One fixation / idt: event count == 1 — got 1
[PASS] One fixation: fixation counts match — ihmm=1, idt=1
[PASS] One fixation: min pairwise overlap >= 0.9 — worst overlap=0.99


True

## Case 3 — One big fixation

The gaze position never moves at all: a single constant `(x, y)` for the *entire* recording, starting from sample 0. This checks that both algorithms correctly treat a fixation that begins right at the start of the recording as one single event covering (almost) the full trial, rather than splitting it or dropping the boundary samples.

In [7]:
length_big_fix = 200
positions_big_fix = np.tile(np.array([[75., 75.]]), (length_big_fix, 1))

velocities_big_fix = pos2vel(positions_big_fix, sampling_rate=sampling_rate)

ihmmEvents = ihmm(velocities_big_fix, reestimation=True, name="ihmm_fix")
idtEvents = idt(positions_big_fix, name="idt_fix")

show("IHMM — one big fixation", ihmmEvents)
show("IDT — one big fixation", idtEvents)


--- IHMM — one big fixation (1 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""ihmm_fix""",0,199,199


--- IDT — one big fixation (1 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""idt_fix""",0,199,199


In [8]:
assert_event_count("One big fixation / ihmm", ihmmEvents, 1)
assert_event_count("One big fixation / idt", idtEvents, 1)

# The single fixation should cover (nearly) the whole recording.
min_span = length_big_fix * 0.9
for label, events in (("ihmm", ihmmEvents), ("idt", idtEvents)):
    onset, offset = spans(events)[0]
    covered = offset - onset
    ok = covered >= min_span
    record(f"One big fixation / {label} covers >= 90% of trial", ok, f"span={covered}")
    assert ok

assert_algorithms_agree("One big fixation", ihmmEvents, idtEvents, min_overlap=0.95)


[PASS] One big fixation / ihmm: event count == 1 — got 1
[PASS] One big fixation / idt: event count == 1 — got 1
[PASS] One big fixation / ihmm covers >= 90% of trial — span=199
[PASS] One big fixation / idt covers >= 90% of trial — span=199
[PASS] One big fixation: fixation counts match — ihmm=1, idt=1
[PASS] One big fixation: min pairwise overlap >= 0.95 — worst overlap=1.00


True

## Case 4 — Trailing and ending fixations

Two long constant segments back to back, with a single abrupt jump between them: a fixation that starts at sample 0 ("leading"/trailing-edge fixation), a saccade, then a second fixation that runs all the way to the last sample ("ending" fixation). This is the classic edge case for windowed algorithms: boundary fixations must not be clipped, merged away, or dropped just because they touch the start/end of the recording.

In [9]:
length_edges = 200
split_at = 90
positions_edges = step_function(
    length=length_edges,
    steps=[split_at],
    values=[(60., 60.)],
    start_value=(10., 10.),
)

velocities_edges = pos2vel(positions_edges, sampling_rate=sampling_rate)

ihmmEvents = ihmm(velocities_edges, reestimation=True, name="ihmm_fix",minimum_duration=2)
idtEvents = idt(positions_edges, name="idt_fix",minimum_duration=2)

show("IHMM — trailing/ending fixations", ihmmEvents)
show("IDT — trailing/ending fixations", idtEvents)


--- IHMM — trailing/ending fixations (2 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""ihmm_fix""",0,87,87
"""ihmm_fix""",92,199,107


--- IDT — trailing/ending fixations (2 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""idt_fix""",0,90,90
"""idt_fix""",91,199,108


In [10]:
assert_event_count("Trailing/ending fixations / ihmm", ihmmEvents, 2)
assert_event_count("Trailing/ending fixations / idt", idtEvents, 2)

for label, events in (("ihmm", ihmmEvents), ("idt", idtEvents)):
    event_spans = sorted(spans(events), key=lambda s: s[0])
    first_onset = event_spans[0][0]
    last_offset = event_spans[-1][1]
    starts_at_edge = first_onset <= 1  # allow 1 sample of slack (ms at 1000Hz)
    ends_at_edge = last_offset >= (length_edges - 1) * (1000.0 / sampling_rate) - 1
    record(f"Trailing/ending / {label}: first fixation starts at trial start", starts_at_edge, f"onset={first_onset}")
    record(f"Trailing/ending / {label}: last fixation ends at trial end", ends_at_edge, f"offset={last_offset}")
    assert starts_at_edge and ends_at_edge

assert_algorithms_agree("Trailing/ending fixations", ihmmEvents, idtEvents, min_overlap=0.85)


[PASS] Trailing/ending fixations / ihmm: event count == 2 — got 2
[PASS] Trailing/ending fixations / idt: event count == 2 — got 2
[PASS] Trailing/ending / ihmm: first fixation starts at trial start — onset=0
[PASS] Trailing/ending / ihmm: last fixation ends at trial end — offset=199
[PASS] Trailing/ending / idt: first fixation starts at trial start — onset=0
[PASS] Trailing/ending / idt: last fixation ends at trial end — offset=199
[PASS] Trailing/ending fixations: fixation counts match — ihmm=2, idt=2
[PASS] Trailing/ending fixations: min pairwise overlap >= 0.85 — worst overlap=0.97


True

## Case 5 — Toy dataset

Runs both detectors on pymovements' public `ToyDataset` (a small, real, noisy eye-tracking recording). We can't hand-derive an exact expected fixation count for real data, so correctness here means: both algorithms produce a *plausible, comparable* number of fixations on the same recording — not wildly diverging counts. This requires network access to download the dataset the first time it's run.

In [11]:
import pymovements as pm

toy_available = True
try:
    dataset = pm.Dataset("ToyDataset", path="data/ToyDataset")
    dataset.download()
    dataset.load()
except Exception as exc:  # e.g. no network access in this environment
    toy_available = False
    print(f"Skipping toy dataset test — could not download/load ToyDataset ({exc})")


INFO:pymovements.dataset.dataset:
        You are downloading the pymovements Toy Dataset. Please be aware that pymovements does not
        host or distribute any dataset resources and only provides a convenient interface to
        download the public dataset resources that were published by their respective authors.

        Please cite the referenced publication if you intend to use the dataset in your research.
        


pymovements-toy-dataset.zip: 0.00B [00:00, ?B/s]

Checking integrity of pymovements-toy-dataset.zip
Extracting pymovements-toy-dataset.zip to data\ToyDataset\raw


Extracting archive: 100%|██████████| 23/23 [00:00<00:00, 351.32file/s]


Loading gaze files:   0%|          | 0/20 [00:00<?, ?file/s]

In [12]:
if toy_available:
    dataset.pix2deg()
    dataset.pos2vel()

    toy_gaze = dataset.gaze[0]
    toy_positions = toy_gaze.frame.select("position").to_series().to_list()
    toy_velocities = toy_gaze.frame.select("velocity").to_series().to_list()
    
    toy_velocities = [[np.nan,np.nan] if v == [None,None] else v for v in toy_velocities]
    print(toy_velocities)

    ihmmEvents = ihmm(toy_velocities, reestimation=True, name="ihmm_fix")
    idtEvents = idt(np.array(toy_positions), name="idt_fix")

    show("IHMM — toy dataset", ihmmEvents)
    show("IDT — toy dataset", idtEvents)


Applying pix2deg:   0%|          | 0/20 [00:00<?, ?file/s]

Applying pos2vel:   0%|          | 0/20 [00:00<?, ?file/s]

C:\Users\dioni\AppData\Local\Temp\ipykernel_9028\2272514501.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) frame. (Please use Gaze.samples instead. This property will be removed in v0.28.0.) -- Deprecated since version v0.23.0.
  toy_positions = toy_gaze.frame.select("position").to_series().to_list()
C:\Users\dioni\AppData\Local\Temp\ipykernel_9028\2272514501.py:7: DeprecationWarning: Call to deprecated function (or staticmethod) frame. (Please use Gaze.samples instead. This property will be removed in v0.28.0.) -- Deprecated since version v0.23.0.
  toy_velocities = toy_gaze.frame.select("velocity").to_series().to_list()


[[nan, nan], [nan, nan], [1.610193876635968, -5.256267134894206], [0.40254846947836614, -4.447464804337405], [0.40256128509152944, -3.234461575639891], [2.0128192346839193, -0.8086153945822497], [4.025683299096508, 2.8301699070576447], [4.428327837753857, 5.256090882933624], [3.2206631899232008, 4.851847297117887], [1.6103540034346033, 3.2346218358512147], [-2.9605947323337506e-13, 1.6173429545135083], [-0.8051866035222824, 1.2130245617842186], [-0.8051866035222824, 0.4043450779009916], [-2.9605947323337506e-13, -2.0216666805132157], [1.2077863047821324, -4.447630381693379], [2.818223494026976, -4.851922066401096], [5.234007179625817, -3.6388601004884906], [5.636709214038878, -3.234440191488848], [1.6105011369136335, -4.4472403003113685], [-3.6236083758653947, -5.25574337369535], [-4.428890916877333, -3.234269053799051], [-2.5574974138464768e-05, 0.8085672634496888], [2.01309445402309, 4.851446367808579], [-1.207805499442216, 5.255770120675685], [-4.831285956950518, 2.8300576409359124]

name,onset,offset,duration
str,i64,i64,i64
"""ihmm_fix""",2,176,174
"""ihmm_fix""",207,401,194
"""ihmm_fix""",448,590,142
"""ihmm_fix""",646,868,222
"""ihmm_fix""",916,1025,109
…,…,…,…
"""ihmm_fix""",15309,15512,203
"""ihmm_fix""",15578,15771,193
"""ihmm_fix""",15988,16140,152


--- IDT — toy dataset (77 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""idt_fix""",0,188,188
"""idt_fix""",189,407,218
"""idt_fix""",412,548,136
"""idt_fix""",606,874,268
"""idt_fix""",878,1034,156
…,…,…,…
"""idt_fix""",16218,16407,189
"""idt_fix""",16408,16551,143
"""idt_fix""",16552,16666,114


In [13]:
if toy_available:
    ihmm_count, idt_count = n_events(ihmmEvents), n_events(idtEvents)
    larger, smaller = max(ihmm_count, idt_count), max(min(ihmm_count, idt_count), 1)
    relative_diff = (larger - smaller) / smaller
    ok = relative_diff <= 0.5  # allow up to 50% relative disagreement on real, noisy data
    record(
        "Toy dataset: fixation counts are comparable",
        ok,
        f"ihmm={ihmm_count}, idt={idt_count}, relative diff={relative_diff:.2f}",
    )
else:
    record("Toy dataset: fixation counts are comparable", None, "skipped (no network access)")


[PASS] Toy dataset: fixation counts are comparable — ihmm=70, idt=77, relative diff=0.10


## Case 6 — Regular (realistic, noisy) dataset

A synthetic but realistic reading-like trace: several fixations of varying duration connected by saccades, with small Gaussian jitter added on top to emulate eye-tracker measurement noise. Built with a fixed random seed so this test is reproducible. As with the toy dataset, we don't assert an exact fixation count (jitter can push either algorithm's threshold either way) but require the two algorithms to broadly agree.

In [14]:
rng = np.random.default_rng(seed=42)

fixation_targets = [
    (0., 0.), (120., 5.), (240., -10.), (360., 15.), (480., 0.),
    (600., 20.), (720., -15.), (840., 10.), (960., 0.),
]
fixation_len = 150  # samples held per fixation target

length_realistic = fixation_len * len(fixation_targets)
steps_realistic = [fixation_len * i for i in range(1, len(fixation_targets))]

positions_realistic = step_function(
    length=length_realistic,
    steps=steps_realistic,
    values=fixation_targets[1:],
    start_value=fixation_targets[0],
)

# Small measurement jitter, well below any reasonable dispersion/velocity threshold.
jitter = rng.normal(loc=0.0, scale=0.05, size=positions_realistic.shape)
positions_realistic = positions_realistic + jitter

velocities_realistic = pos2vel(positions_realistic, sampling_rate=sampling_rate)

ihmmEvents = ihmm(velocities_realistic, reestimation=True, name="ihmm_fix")
idtEvents = idt(positions_realistic, name="idt_fix")

show("IHMM — regular dataset", ihmmEvents)
show("IDT — regular dataset", idtEvents)


--- IHMM — regular dataset (9 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""ihmm_fix""",0,147,147
"""ihmm_fix""",152,297,145
"""ihmm_fix""",302,447,145
"""ihmm_fix""",452,597,145
"""ihmm_fix""",602,747,145
"""ihmm_fix""",752,897,145
"""ihmm_fix""",902,1047,145
"""ihmm_fix""",1052,1197,145
"""ihmm_fix""",1202,1349,147


--- IDT — regular dataset (9 event(s)) ---


name,onset,offset,duration
str,i64,i64,i64
"""idt_fix""",0,150,150
"""idt_fix""",151,300,149
"""idt_fix""",301,450,149
"""idt_fix""",451,600,149
"""idt_fix""",601,750,149
"""idt_fix""",751,900,149
"""idt_fix""",901,1050,149
"""idt_fix""",1051,1200,149
"""idt_fix""",1201,1349,148


In [15]:
expected_fixations = len(fixation_targets)

for label, events in (("ihmm", ihmmEvents), ("idt", idtEvents)):
    actual = n_events(events)
    ok = abs(actual - expected_fixations) <= 1  # allow off-by-one at the boundaries
    record(f"Regular dataset / {label}: ~{expected_fixations} fixations detected", ok, f"got {actual}")
    assert ok

assert_algorithms_agree("Regular dataset", ihmmEvents, idtEvents, min_overlap=0.6)


[PASS] Regular dataset / ihmm: ~9 fixations detected — got 9
[PASS] Regular dataset / idt: ~9 fixations detected — got 9
[PASS] Regular dataset: fixation counts match — ihmm=9, idt=9
[PASS] Regular dataset: min pairwise overlap >= 0.6 — worst overlap=0.97


True

## Summary

In [16]:
import pandas as pd

summary = pd.DataFrame(test_results)
display(summary)

n_run = summary["passed"].notna().sum()
n_passed = (summary["passed"] == True).sum()
n_failed = (summary["passed"] == False).sum()
n_skipped = summary["passed"].isna().sum()
print(f"{n_passed}/{n_run} checks passed" + (f", {n_skipped} skipped" if n_skipped else ""))


,test,passed,detail
0,No fixations / ihmm: event count == 0,True,got 0
1,No fixations / idt: event count == 0,True,got 0
2,One fixation / ihmm: event count == 1,True,got 1
3,One fixation / idt: event count == 1,True,got 1
4,One fixation: fixation counts match,True,"ihmm=1, idt=1"
5,One fixation: min pairwise overlap >= 0.9,True,worst overlap=0.99
6,One big fixation / ihmm: event count == 1,True,got 1
7,One big fixation / idt: event count == 1,True,got 1
8,One big fixation / ihmm covers >= 90% of trial,True,span=199
9,One big fixation / idt covers >= 90% of trial,True,span=199


25/25 checks passed
